In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

def top_counts_series(df, col, topn=20, truncate_val=None):
    if col not in df.columns:
        return pd.Series(dtype=int)
    vc = df[col].value_counts().head(topn)
    if truncate_val:
        def trunc(v):
            s = str(v)
            return s if len(s) <= truncate_val else s[:truncate_val-3] + '...'
        vc.index = [trunc(i) for i in vc.index]
    return vc

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df  = pd.read_csv('trace.csv')
log_df    = pd.read_csv('log.csv')
error_df  = pd.read_csv('error_logs.csv')

# Helper to build compact one-row summary DataFrame
def build_summary(df, name, name_col=None, sample_col=None):
    rows = df.shape[0]
    min_ts = df['timestamp'].min() if 'timestamp' in df.columns else None
    max_ts = df['timestamp'].max() if 'timestamp' in df.columns else None
    min_ts_s = to_tz_str(min_ts) if min_ts is not None else None
    max_ts_s = to_tz_str(max_ts) if max_ts is not None else None

    top_cmdb = top_counts_series(df, 'cmdb_id', topn=20)
    # format as "id(count)" joined string
    top_cmdb_str = ', '.join([f"{idx}({cnt})" for idx, cnt in top_cmdb.items()])

    if name_col:
        top_names = top_counts_series(df, name_col, topn=20, truncate_val=80)
        top_names_str = ', '.join([f"{idx}({cnt})" for idx,cnt in top_names.items()])
    else:
        top_names_str = ''

    # For error_logs, sample messages
    if sample_col:
        top_msgs = top_counts_series(df, sample_col, topn=20, truncate_val=120)
        top_msgs_str = ', '.join([f"{idx}({cnt})" for idx,cnt in top_msgs.items()])
    else:
        top_msgs_str = ''

    summary = pd.DataFrame([{
        'file': name,
        'rows': rows,
        'min_timestamp_utc+8': min_ts_s,
        'max_timestamp_utc+8': max_ts_s,
        'top_cmdb_id_top20': top_cmdb_str if top_cmdb_str else None,
        'top_names_top20': top_names_str if top_names_str else None,
        'top_messages_sample_top20': top_msgs_str if top_msgs_str else None
    }])
    sample_head = df.head(5)
    return summary, top_cmdb, (top_names if name_col else pd.Series(dtype=int)), (top_msgs if sample_col else pd.Series(dtype=int)), sample_head

metric_summary, metric_top_cmdb, metric_top_kpis, _, metric_head = build_summary(metric_df, 'metric.csv', name_col='kpi_name')
trace_summary, trace_top_cmdb, trace_top_traces, _, trace_head = build_summary(trace_df, 'trace.csv', name_col='trace_name')
log_summary, log_top_cmdb, log_top_logs, _, log_head = build_summary(log_df, 'log.csv', name_col='log_name')
error_summary, error_top_cmdb, _, error_top_messages, error_head = build_summary(error_df, 'error_logs.csv', sample_col='message')

# Compact combined summary table (one row per file)
combined_summary = pd.concat([metric_summary, trace_summary, log_summary, error_summary], ignore_index=True)

# Display a few compact helper top lists (limited to 20) rather than full frames
metric_top_cmdb, metric_top_kpis.head(20), metric_head, trace_top_cmdb, trace_top_traces.head(20), trace_head, log_top_cmdb, log_top_logs.head(20), log_head, error_top_cmdb, error_top_messages.head(20), error_head, combined_summary

```
Out[1]:
```
# Summary string in plain English based on the loaded telemetry files (timestamps shown in UTC+8)
summary_text = (
    "Summary (timestamps in UTC+8):\n\n"
    "metric.csv:\n"
    "- Total rows: 145,812\n"
    "- Time range: 2022-03-21 01:30:00 to 2022-03-21 01:59:00\n"
    "- Top cmdb_id (by row count, top 10): adservice(10230), adservice2(9990), frontend2-0(5040), frontend-0(4800), frontend-1(4680), frontend-2(4500), checkoutservice-0(3780), checkoutservice-1(3540), checkoutservice-2(3540), productcatalogservice-0(3480)\n"
    "- Top kpi_name examples (top entries): container.node-5.container_network_receive_MB.eth0, container.node-5.container_network_receive_packets.eth0, container.node-5.container_network_transmit_packets_dropped.eth0, container.node-5.container_network_receive_errors.eth0, container.node-5.container_network_transmit_errors.eth0, ... (many container/node-level metrics)\n"
    "- Sample first 5 rows: see metric head (timestamp 1647797400 for adservice entries)\n\n"
    "trace.csv:\n"
    "- Total rows: 28,944\n"
    "- Time range: 2022-03-21 01:30:00 to 2022-03-21 01:59:00\n"
    "- Top cmdb_id (top 10): frontend-0(2584), frontend-2(2504), frontend-1(2496), checkoutservice-2(1480), checkoutservice-1(1464), checkoutservice-0(1460), productcatalogservice-1(1056), frontend2-0(1056), productcatalogservice-0(1048), productcatalogservice-2(1044)\n"
    "- Top trace names (examples): trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count, trace.from_frontend-2.row_count, ...\n"
    "- Sample first 5 rows: see trace head (adservice-0 traces at timestamp 1647797400)\n\n"
    "log.csv:\n"
    "- Total rows: 1,720\n"
    "- Time range: 2022-03-21 01:30:00 to 2022-03-21 01:59:00\n"
    "- Top cmdb_id (examples): adservice-0(60), adservice-1(60), adservice-2(60), cartservice-0(60), cartservice-1(60), ...\n"
    "- Top log names: log.error_count(860), log.row_count(860)\n"
    "- Sample first 5 rows: see log head (adservice-* log entries at timestamp 1647797400)\n\n"
    "error_logs.csv:\n"
    "- Total rows: 543\n"
    "- Time range: 2022-03-21 01:30:04 to 2022-03-21 01:59:59\n"
    "- Top cmdb_id (top): frontend-0(233), frontend-1(155), frontend-2(155)\n"
    "- Top message sample (all or majority): \"severity: warning, message: failed to retrieve ads\" (543 occurrences)\n"
    "- Sample first 5 rows: see error_logs head (frontend-* messages around 1647797404–1647797412)\n\n"
    "Notes:\n"
    "- All timestamps converted to Asia/Shanghai (UTC+8).\n"
    "- Top lists above are truncated for brevity (examples and top entries shown)."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(cmdb_id
adservice                   10230
adservice2                   9990
frontend2-0                  5040
frontend-0                   4800
frontend-1                   4680
frontend-2                   4500
checkoutservice-0            3780
checkoutservice-1            3540
checkoutservice-2            3540
productcatalogservice-0      3480
productcatalogservice-2      3480
productcatalogservice-1      3360
checkoutservice2-0           3300
adservice2-0                 3060
productcatalogservice2-0     2880
cartservice-1                2880
cartservice-0                2760
cartservice-2                2640
cartservice2-0               2640
adservice-0                  2580
Name: count, dtype: int64, container.node-5.container_network_receive_MB.eth0                  1050
container.node-5.container_network_receive_packets.eth0             1050
container.node-5.container_network_transmit_packets_dropped.eth0    1050
container.node-5.container_network_receive_errors.eth0              1050
container.node-5.container_network_transmit_errors.eth0             1050
container.node-5.container_network_transmit_MB.eth0                 1050
container.node-5.container_network_receive_packets_dropped.eth0     1050
container.node-5.container_network_transmit_packets.eth0            1050
container.node-5.container_memory_failures.container.pgfault         990
container.node-5.container_memory_rss                                990
container.node-5.container_memory_failures.hierarchy.pgfault         990
container.node-5.container_memory_swap                               990
container.node-5.container_tasks_state.iowaiting                     990
container.node-5.container_spec_memory_swap_limit_MB                 990
container.node-5.container_fs_writes_merged./dev/vda1                990
container.node-5.container_spec_cpu_period                           990
container.node-5.container_threads_max                               990
container.node-5.container_ulimits_soft.max_open_files               990
container.node-5.container_tasks_state.stopped                       990
container.node-5.container_tasks_state.sleeping                      990
Name: count, dtype: int64,     timestamp    cmdb_id        kpi_name       value
0  1647797400  adservice  app.grpc.count   70.000000
1  1647797400  adservice    app.grpc.mrt    0.002493
2  1647797400  adservice     app.grpc.rr  100.000000
3  1647797400  adservice     app.grpc.sr   77.142857
4  1647797400  adservice  app.http.count   24.000000, cmdb_id
frontend-0                 2584
frontend-2                 2504
frontend-1                 2496
checkoutservice-2          1480
checkoutservice-1          1464
checkoutservice-0          1460
productcatalogservice-1    1056
frontend2-0                1056
productcatalogservice-0    1048
productcatalogservice-2    1044
checkoutservice2-0          864
recommendationservice-2     840
recommendationservice-1     840
recommendationservice-0     840
cartservice-0               700
cartservice-2               692
cartservice-1               680
currencyservice-1           612
currencyservice-0           608
currencyservice-2           600
Name: count, dtype: int64, trace.from_frontend-0.duration_mean           586
trace.from_frontend-0.duration_p95            586
trace.from_frontend-0.error_rate              586
trace.from_frontend-0.row_count               586
trace.from_frontend-2.row_count               566
trace.from_frontend-2.error_rate              566
trace.from_frontend-2.duration_p95            566
trace.from_frontend-2.duration_mean           566
trace.from_frontend-1.row_count               564
trace.from_frontend-1.error_rate              564
trace.from_frontend-1.duration_p95            564
trace.from_frontend-1.duration_mean           564
trace.self.row_count                          448
trace.self.error_rate                         448
trace.self.duration_p95                       448
trace.self.duration_mean                      448
trace.from_checkoutservice-2.duration_mean    292
trace.from_checkoutservice-2.duration_p95     292
trace.from_checkoutservice-2.error_rate       292
trace.from_checkoutservice-2.row_count        292
Name: count, dtype: int64,     timestamp      cmdb_id                           trace_name     value
0  1647797400  adservice-0  trace.from_frontend-0.duration_mean  0.000017
1  1647797400  adservice-0   trace.from_frontend-0.duration_p95  0.000021
2  1647797400  adservice-0     trace.from_frontend-0.error_rate  0.000000
3  1647797400  adservice-0      trace.from_frontend-0.row_count  8.000000
4  1647797400  adservice-0  trace.from_frontend-1.duration_mean  0.000013, cmdb_id
adservice-0                60
adservice-1                60
adservice-2                60
cartservice-0              60
cartservice-1              60
cartservice-2              60
currencyservice-1          60
frontend-0                 60
currencyservice-0          60
currencyservice-2          60
productcatalogservice-2    60
productcatalogservice-0    60
productcatalogservice-1    60
frontend-2                 60
frontend-1                 60
shippingservice-1          60
shippingservice-2          60
shippingservice-0          60
recommendationservice-0    60
recommendationservice-1    60
Name: count, dtype: int64, log.error_count    860
log.row_count      860
Name: count, dtype: int64,     timestamp      cmdb_id         log_name  value
0  1647797400  adservice-0  log.error_count    0.0
1  1647797400  adservice-0    log.row_count  430.0
2  1647797400  adservice-1  log.error_count    0.0
3  1647797400  adservice-1    log.row_count   32.0
4  1647797400  adservice-2  log.error_count    0.0, cmdb_id
frontend-0    233
frontend-1    155
frontend-2    155
Name: count, dtype: int64, severity: warning, message: failed to retrieve ads    543
Name: count, dtype: int64,     timestamp     cmdb_id                                            message
0  1647797404  frontend-0  severity: warning, message: failed to retrieve...
1  1647797404  frontend-1  severity: warning, message: failed to retrieve...
2  1647797406  frontend-2  severity: warning, message: failed to retrieve...
3  1647797411  frontend-1  severity: warning, message: failed to retrieve...
4  1647797412  frontend-2  severity: warning, message: failed to retrieve...,              file    rows      min_timestamp_utc+8      max_timestamp_utc+8                                  top_cmdb_id_top20                                    top_names_top20                          top_messages_sample_top20
0      metric.csv  145812  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  adservice(10230), adservice2(9990), frontend2-...  container.node-5.container_network_receive_MB....                                               None
1       trace.csv   28944  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  frontend-0(2584), frontend-2(2504), frontend-1...  trace.from_frontend-0.duration_mean(586), trac...                                               None
2         log.csv    1720  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  adservice-0(60), adservice-1(60), adservice-2(...           log.error_count(860), log.row_count(860)                                               None
3  error_logs.csv     543  2022-03-21 01:30:04 CST  2022-03-21 01:59:59 CST  frontend-0(233), frontend-1(155), frontend-2(155)                                               None  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

# Aggregate metric time series by (cmdb_id, kpi_name) across the full metric_df (no time filtering)
# Reuse existing metric_df, tz, to_tz_str from previous session.

agg = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    mean=('value', 'mean'),
    min=('value', 'min'),
    max=('value', 'max'),
    p95=('value', lambda x: x.quantile(0.95)),
    p5=('value', lambda x: x.quantile(0.05))
).reset_index()

# Format timestamps to UTC+8 strings for readability (store alongside numeric timestamps)
agg['min_timestamp_utc+8'] = agg['min_timestamp'].apply(lambda x: to_tz_str(x))
agg['max_timestamp_utc+8'] = agg['max_timestamp'].apply(lambda x: to_tz_str(x))

# Keep full aggregation table in a kernel variable for later use
metric_agg_df = agg.copy()

# Prepare a compact top-20 table ordered by count (highest first) with requested columns
display_cols = [
    'cmdb_id', 'kpi_name', 'count',
    'min_timestamp_utc+8', 'max_timestamp_utc+8',
    'mean', 'min', 'max', 'p95', 'p5'
]
top20 = metric_agg_df.sort_values('count', ascending=False)[display_cols].head(20)

# Round numeric columns for compact display
for c in ['mean', 'min', 'max', 'p95', 'p5']:
    top20[c] = top20[c].round(6)

# Return the stored full aggregation shape (compact) and the top20 rows
metric_agg_df.shape, top20

```
Out[2]:
```
summary_text = (
    "Aggregation summary (metric.csv -> grouped by (cmdb_id, kpi_name)):\n\n"
    "- Total groups (unique cmdb_id + kpi_name): 4,870\n"
    "- Computed metrics per group: count, min_timestamp, max_timestamp, mean, min, max, P95, P5\n"
    "- Time span seen in top groups: 2022-03-21 01:30:00 to 2022-03-21 01:59:00 (UTC+8)\n"
    "- Typical group size: many groups have 30 samples (top groups shown earlier all have count=30)\n"
    "- Example top groups by count:\n"
    "  * shippingservice2-0 / mesh.source.shippingservice2.jaeger-collector.... (count=30)\n"
    "  * adservice / app.grpc.count (count=30)\n"
    "  * adservice / app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count (count=30)\n"
    "  * multiple shippingservice2-0 container.node-5.* metrics (network, memory, etc.) (count=30)\n\n"
    "- The full aggregation table is stored in the kernel variable: metric_agg_df\n"
    "- Use metric_agg_df for further analysis or to extract specific groups/time windows.\n"
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

((4870, 12),                  cmdb_id                                           kpi_name  count      min_timestamp_utc+8      max_timestamp_utc+8          mean           min           max           p95            p5
4869  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  8.042917e+02  3.980000e+02  1.144250e+03  1.044750e+03  4.203875e+02
0              adservice                                     app.grpc.count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  7.653333e+01  6.400000e+01  8.800000e+01  8.600000e+01  6.800000e+01
1              adservice                                       app.grpc.mrt     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  2.483000e-03  2.316000e-03  2.979000e-03  2.626000e-03  2.370000e-03
2              adservice                                        app.grpc.rr     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
3              adservice                                        app.grpc.sr     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  7.653355e+01  7.341772e+01  8.026316e+01  8.002312e+01  7.347886e+01
4              adservice                                     app.http.count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  2.400000e+01  2.400000e+01  2.400000e+01  2.400000e+01  2.400000e+01
4830  shippingservice2-0  container.node-5.container_network_receive_MB....     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  4.561500e-02  2.100000e-02  3.017450e-01  1.743420e-01  2.229400e-02
4831  shippingservice2-0  container.node-5.container_network_receive_err...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4832  shippingservice2-0  container.node-5.container_network_receive_pac...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  2.954000e+02  2.220000e+02  4.390000e+02  3.391250e+02  2.392000e+02
4833  shippingservice2-0  container.node-5.container_network_receive_pac...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4834  shippingservice2-0  container.node-5.container_network_transmit_MB...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  4.545160e-01  2.301460e-01  6.916770e-01  5.723840e-01  3.316730e-01
4835  shippingservice2-0  container.node-5.container_network_transmit_er...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4836  shippingservice2-0  container.node-5.container_network_transmit_pa...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  2.009500e+02  1.510000e+02  3.120000e+02  2.393750e+02  1.598750e+02
4837  shippingservice2-0  container.node-5.container_network_transmit_pa...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4822  shippingservice2-0  container.node-5.container_memory_failures.hie...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  1.963333e+01  0.000000e+00  6.100000e+01  4.830000e+01  0.000000e+00
4823  shippingservice2-0  container.node-5.container_memory_failures.hie...     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4824  shippingservice2-0      container.node-5.container_memory_mapped_file     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  4.943872e+05  4.915200e+05  4.956160e+05  4.956160e+05  4.915200e+05
4825  shippingservice2-0     container.node-5.container_memory_max_usage_MB     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  1.280117e+02  1.280117e+02  1.280117e+02  1.280117e+02  1.280117e+02
4826  shippingservice2-0              container.node-5.container_memory_rss     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  1.443970e+07  1.424179e+07  1.460634e+07  1.459405e+07  1.425029e+07
4827  shippingservice2-0             container.node-5.container_memory_swap     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# reuse existing variables: metric_df, metric_agg_df, to_tz_str, tz if present; otherwise redefine tz/to_tz_str
tz = pytz.timezone('Asia/Shanghai')

def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Define incident window in UTC+8 and convert to epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 01:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 02:00:00', tz=tz)
start_ts = int(start_local.tz_convert(pytz.utc).timestamp())
end_ts   = int(end_local.tz_convert(pytz.utc).timestamp())

# Filter metric_df to the window
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# Compute within-window aggregates per group (only groups that have data in window)
win_agg = window_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count_in_window=('value', 'size'),
    min_in_window=('value', 'min'),
    max_in_window=('value', 'max')
).reset_index()

# For timestamp_of_min (earliest timestamp when value == min_in_window) and timestamp_of_max (earliest when value == max_in_window)
# Merge to find matching rows, then group to get earliest timestamp
if not win_agg.empty:
    min_match = window_df.merge(win_agg[['cmdb_id','kpi_name','min_in_window']], on=['cmdb_id','kpi_name'])
    min_match = min_match[min_match['value'] == min_match['min_in_window']]
    ts_min = min_match.groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'timestamp_of_min'})

    max_match = window_df.merge(win_agg[['cmdb_id','kpi_name','max_in_window']], on=['cmdb_id','kpi_name'])
    max_match = max_match[max_match['value'] == max_match['max_in_window']]
    ts_max = max_match.groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'timestamp_of_max'})

    win_agg = win_agg.merge(ts_min, on=['cmdb_id','kpi_name'], how='left')
    win_agg = win_agg.merge(ts_max, on=['cmdb_id','kpi_name'], how='left')
else:
    # empty window: create empty columns consistent with schema
    win_agg['timestamp_of_min'] = pd.Series(dtype='float64')
    win_agg['timestamp_of_max'] = pd.Series(dtype='float64')

# Now start from the global groups in metric_agg_df to include groups with zero counts in window
global_groups = metric_agg_df[['cmdb_id', 'kpi_name', 'p95', 'p5']].copy()
metric_window_agg_df = global_groups.merge(win_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill missing counts with 0; leave min/max as NaN when absent
metric_window_agg_df['count_in_window'] = metric_window_agg_df['count_in_window'].fillna(0).astype(int)

# Compute breach ratios with rules:
# upward_breach_ratio = (max_in_window - P95)/P95 if P95>0 and max_in_window>P95 else 0
# downward_breach_ratio = (P5 - min_in_window)/P5 if P5>0 and min_in_window<P5 else 0
def compute_upward(row):
    p95 = row['p95']
    maxv = row['max_in_window']
    if pd.isna(p95) or p95 == 0 or pd.isna(maxv):
        return 0.0
    return float((maxv - p95) / p95) if maxv > p95 else 0.0

def compute_downward(row):
    p5 = row['p5']
    minv = row['min_in_window']
    if pd.isna(p5) or p5 == 0 or pd.isna(minv):
        return 0.0
    return float((p5 - minv) / p5) if minv < p5 else 0.0

metric_window_agg_df['upward_breach_ratio'] = metric_window_agg_df.apply(compute_upward, axis=1)
metric_window_agg_df['downward_breach_ratio'] = metric_window_agg_df.apply(compute_downward, axis=1)
metric_window_agg_df['severity'] = metric_window_agg_df[['upward_breach_ratio','downward_breach_ratio']].max(axis=1)

# Prepare display columns and formatting
display_df = metric_window_agg_df.copy()

# Convert timestamps to readable UTC+8 strings for display
display_df['timestamp_of_min_str'] = display_df['timestamp_of_min'].apply(lambda x: to_tz_str(x) if not pd.isna(x) else None)
display_df['timestamp_of_max_str'] = display_df['timestamp_of_max'].apply(lambda x: to_tz_str(x) if not pd.isna(x) else None)

# Round numeric columns for compactness
for c in ['min_in_window','max_in_window','p95','p5','upward_breach_ratio','downward_breach_ratio','severity']:
    if c in display_df.columns:
        display_df[c] = display_df[c].round(6)

# Select required columns in requested order
result_cols = [
    'cmdb_id', 'kpi_name', 'count_in_window',
    'min_in_window', 'timestamp_of_min_str',
    'max_in_window', 'timestamp_of_max_str',
    'p95', 'p5', 'upward_breach_ratio', 'downward_breach_ratio'
]

# Sort by severity desc and take top 20
top20_severe = display_df.sort_values('severity', ascending=False)[result_cols].head(20)

# Final compact output: store full table in metric_window_agg_df and display top20
metric_window_agg_df.shape, top20_severe

```
Out[3]:
```
summary_text = (
    "Incident window: 2022-03-21 01:30:00 to 2022-03-21 02:00:00 (UTC+8).\n\n"
    "Overview:\n"
    "- Groups analyzed (global): 4,870. Window-level aggregates stored in metric_window_agg_df.\n"
    "- Top-20 groups by severity (max of upward/downward breach) show clear anomalies concentrated on:\n\n"
    "Key findings (top examples):\n"
    "1) Widespread large upward breaches on container.node-5.container_network_receive_MB.* across many services\n"
    "   - Affected cmdb_id examples: paymentservice-1, emailservice2-0, emailservice-1, emailservice-2, shippingservice-2, adservice-2, checkoutservice-1/0/2, adservice-0, etc.\n"
    "   - Each group has count_in_window = 30, global P95 ≈ 0.02–0.09, but max_in_window ≈ 0.55–0.62 -> upward_breach_ratio ~6–21 (very large).\n"
    "   - Interpretation: sudden large spikes in received network MB on node-5 (and one on node-6) for many containers.\n\n"
    "2) JVM / runtime and CPU throttling spikes (examples):\n"
    "   - adservice2 runtime.jvm_classes_loaded: min 0, max 0.8333, P95 0.09167 -> upward ratio ≈ 8.09.\n"
    "   - adservice-0 container_cpu_cfs_throttled_* shows upward ratio ≈ 5.08.\n\n"
    "3) Node-level resource anomalies:\n"
    "   - node-1 system.net.tcp.retrans_segs: max 26.54 vs global P95 4.0955 -> upward ratio ≈ 5.48 (TCP retransmission spike).\n"
    "   - node-4 system.io.w_s (disk write/sec): max 11.5 vs P95 1.775 -> upward ratio ≈ 5.48 (write I/O spike).\n\n"
    "Summary conclusion:\n"
    "- The dominant signal is large, coordinated increases in network receive throughput (container.node-5.*) across many services during the incident window, plus node-level TCP retransmission and disk write I/O spikes.\n"
    "- These point to a network throughput/congestion event on node-5 (and related node-level resource stress) as the primary anomaly to investigate further.\n\n"
    "Notes:\n"
    "- The full window-joined table is in kernel variable: metric_window_agg_df. Top offending groups (20 rows) were returned for quick inspection."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

((4870, 12),                       cmdb_id                                           kpi_name  count_in_window  min_in_window     timestamp_of_min_str  max_in_window     timestamp_of_max_str       p95        p5  upward_breach_ratio  downward_breach_ratio
3584         paymentservice-1  container.node-5.container_network_receive_MB....               30       0.017473  2022-03-21 01:52:00 CST       0.551788  2022-03-21 01:54:00 CST  0.025004  0.017913            21.068376               0.024561
2452          emailservice2-0  container.node-5.container_network_receive_MB....               30       0.019201  2022-03-21 01:47:00 CST       0.549767  2022-03-21 01:37:00 CST  0.025581  0.019602            20.491024               0.020474
2304           emailservice-1  container.node-5.container_network_receive_MB....               30       0.018292  2022-03-21 01:37:00 CST       0.550180  2022-03-21 01:33:00 CST  0.025608  0.019308            20.484377               0.052619
2378           emailservice-2  container.node-5.container_network_receive_MB....               30       0.019011  2022-03-21 01:57:00 CST       0.548522  2022-03-21 01:53:00 CST  0.025781  0.019443            20.275826               0.022212
4750        shippingservice-2  container.node-5.container_network_receive_MB....               30       0.022893  2022-03-21 01:43:00 CST       0.558463  2022-03-21 01:48:00 CST  0.031204  0.023409            16.896899               0.022021
553               adservice-2  container.node-5.container_network_receive_MB....               30       0.026631  2022-03-21 01:55:00 CST       0.575527  2022-03-21 01:36:00 CST  0.037824  0.028656            14.216075               0.070679
1572        checkoutservice-1  container.node-5.container_network_receive_MB....               30       0.021163  2022-03-21 01:59:00 CST       0.563397  2022-03-21 01:52:00 CST  0.040089  0.023939            13.053754               0.115973
1446        checkoutservice-0  container.node-5.container_network_receive_MB....               30       0.025004  2022-03-21 01:39:00 CST       0.562598  2022-03-21 01:35:00 CST  0.040681  0.025295            12.829649               0.011494
1808       checkoutservice2-0  container.node-5.container_network_receive_MB....               30       0.021874  2022-03-21 01:36:00 CST       0.561169  2022-03-21 01:45:00 CST  0.043126  0.024240            12.012201               0.097589
381               adservice-0  container.node-5.container_network_receive_MB....               30       0.028329  2022-03-21 01:44:00 CST       0.580233  2022-03-21 01:31:00 CST  0.047205  0.031527            11.291869               0.101435
871                adservice2                         runtime.jvm_classes_loaded               30       0.000000  2022-03-21 01:30:00 CST       0.833333  2022-03-21 01:47:00 CST  0.091667  0.000000             8.090909               0.000000
1998        currencyservice-1  container.node-5.container_network_receive_MB....               30       0.046943  2022-03-21 01:44:00 CST       0.598571  2022-03-21 01:50:00 CST  0.068484  0.050653             7.740254               0.073242
2074        currencyservice-2  container.node-5.container_network_receive_MB....               30       0.047760  2022-03-21 01:33:00 CST       0.580750  2022-03-21 01:31:00 CST  0.068324  0.052505             7.499990               0.090359
1922        currencyservice-0  container.node-5.container_network_receive_MB....               30       0.047747  2022-03-21 01:46:00 CST       0.591809  2022-03-21 01:56:00 CST  0.073419  0.049314             7.060668               0.031769
4322  recommendationservice-1  container.node-5.container_network_receive_MB....               30       0.056932  2022-03-21 01:47:00 CST       0.591435  2022-03-21 01:38:00 CST  0.076993  0.059979             6.681642               0.050795
2150       currencyservice2-0  container.node-6.container_network_receive_MB....               30       0.049405  2022-03-21 01:32:00 CST       0.621909  2022-03-21 01:52:00 CST  0.089215  0.050861             5.970869               0.028623
3164                   node-1                        system.net.tcp.retrans_segs               30       0.390000  2022-03-21 01:32:00 CST      26.540000  2022-03-21 01:48:00 CST  4.095500  0.394500             5.480283               0.011407
3321                   node-4                                      system.io.w_s               30       0.000000  2022-03-21 01:30:00 CST      11.500000  2022-03-21 01:56:00 CST  1.775000  0.000000             5.478873               0.000000
343               adservice-0  container.node-5.container_cpu_cfs_throttled_s...               30       0.000000  2022-03-21 01:30:00 CST       2.464843  2022-03-21 01:37:00 CST  0.405317  0.000000             5.081268               0.000000
3340                   node-4                        system.net.tcp.retrans_segs               30       0.000000  2022-03-21 01:30:00 CST       0.180000  2022-03-21 01:31:00 CST  0.030000  0.000000             5.000000               0.000000)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing variables: metric_df, metric_agg_df, metric_window_agg_df, tz, to_tz_str, window_df
tz = pytz.timezone('Asia/Shanghai')

# 1) Select groups where upward_breach_ratio > 1 and kpi_name matches conditions
cond_kpi = (
    metric_window_agg_df['kpi_name'].str.contains('container_network_receive_MB', na=False) |
    (metric_window_agg_df['kpi_name'] == 'system.net.tcp.retrans_segs') |
    (metric_window_agg_df['kpi_name'] == 'system.io.w_s')
)
selected_groups = metric_window_agg_df[cond_kpi & (metric_window_agg_df['upward_breach_ratio'] > 1)].copy()

# 2) Merge selected groups (with their global P95 and upward ratio) into the incident-window metric rows
# Use window_df which is metric_df already filtered to the incident window
merged = window_df.merge(
    selected_groups[['cmdb_id', 'kpi_name', 'p95', 'upward_breach_ratio']],
    on=['cmdb_id', 'kpi_name'],
    how='inner'
)

# 3) Find rows where value > group's global P95 (global threshold computed earlier)
breaches = merged[merged['value'] > merged['p95']].copy()

# 4) For each group, get the earliest breach timestamp and the value at that timestamp
if not breaches.empty:
    breaches_sorted = breaches.sort_values(['cmdb_id', 'kpi_name', 'timestamp'])
    earliest = breaches_sorted.groupby(['cmdb_id', 'kpi_name']).first().reset_index()
    # Build output DataFrame with requested columns
    earliest_breaches_df = earliest[['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95', 'upward_breach_ratio']].copy()
    earliest_breaches_df = earliest_breaches_df.rename(columns={
        'timestamp': 'earliest_breach_timestamp_ts',
        'value': 'value_at_earliest',
        'p95': 'global_P95'
    })
    # Add human-readable UTC+8 timestamp
    earliest_breaches_df['earliest_breach_timestamp'] = earliest_breaches_df['earliest_breach_timestamp_ts'].apply(
        lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    )
    # Round numeric columns for compactness
    earliest_breaches_df['value_at_earliest'] = earliest_breaches_df['value_at_earliest'].round(6)
    earliest_breaches_df['global_P95'] = earliest_breaches_df['global_P95'].round(6)
    earliest_breaches_df['upward_breach_ratio'] = earliest_breaches_df['upward_breach_ratio'].round(6)
    # Final ordering by earliest breach timestamp ascending
    earliest_breaches_df = earliest_breaches_df.sort_values('earliest_breach_timestamp_ts', ascending=True)
else:
    # No breaches found -> create empty DataFrame with required columns
    earliest_breaches_df = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'earliest_breach_timestamp_ts', 'value_at_earliest', 'global_P95', 'upward_breach_ratio', 'earliest_breach_timestamp'
    ])

# Keep the full detailed results in kernel variable earliest_breaches_df (as requested)
# Prepare compact view (limit 30 rows) with requested display columns
display_cols = ['cmdb_id', 'kpi_name', 'earliest_breach_timestamp', 'value_at_earliest', 'global_P95', 'upward_breach_ratio']
earliest_breaches_df[display_cols].head(30), earliest_breaches_df.shape

```
Out[4]:
```
```python
# Plain-English summary string based on earliest_breaches_df (timestamps shown in UTC+8)
summary_text = (
    "Selected breach summary (UTC+8):\n\n"
    "- Selection: groups with upward_breach_ratio > 1 and kpi in container_network_receive_MB OR system.net.tcp.retrans_segs OR system.io.w_s.\n"
    "- Groups found: 22 groups with breaches during the incident window (2022-03-21 01:30:00 to 02:00:00 UTC+8).\n\n"
    "Key findings:\n"
    "- Earliest breaches begin at 2022-03-21 01:30:00 (UTC+8) for several services (e.g., currencyservice-2, recommendationservice-1, recommendationservice2-0).\n"
    "- Most breaches are for container.node-5.container_network_receive_MB.* across many services (paymentservice, emailservice, adservice, checkoutservice, currencyservice, recommendationservice, shippingservice, etc.).\n"
    "- Upward breach ratios for these network-receive metrics are very large (roughly ~5 to ~21 times above the group's global P95), indicating sudden, substantial spikes in received network MB on node-5 (one node-6 instance also appears).\n"
    "- Node-level anomalies are present as well: system.net.tcp.retrans_segs and system.io.w_s show early and strong breaches (TCP retransmission spike and disk write I/O spike), with earliest tcp retrans breach at 01:31 and a disk write spike at 01:56 (UTC+8).\n"
    "- Breaches occur across the whole 30-minute window (first at 01:30, last observed around 01:56), suggesting a sustained or recurring resource event rather than a single isolated sample.\n\n"
    "Interpretation:\n"
    "- The dominant signal is coordinated, large network receive throughput spikes on node-5 affecting many containers/services, accompanied by node-level TCP retransmissions and a disk write I/O spike. These point to a network throughput/congestion issue on node-5 (and related node stress) as the primary anomaly to investigate.\n\n"
    "Data note:\n"
    "- Detailed per-group earliest breach rows are stored in kernel variable: earliest_breaches_df (use that for exact timestamps and values)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name earliest_breach_timestamp  value_at_earliest  global_P95  upward_breach_ratio
8          currencyservice-2  container.node-5.container_network_receive_MB....   2022-03-21 01:30:00 CST           0.069791    0.068324             7.499990
19   recommendationservice-1  container.node-5.container_network_receive_MB....   2022-03-21 01:30:00 CST           0.077590    0.076993             6.681642
20  recommendationservice2-0  container.node-6.container_network_receive_MB....   2022-03-21 01:30:00 CST           0.616517    0.106970             4.763439
0                adservice-0  container.node-5.container_network_receive_MB....   2022-03-21 01:31:00 CST           0.580233    0.047205            11.291869
16                    node-4                        system.net.tcp.retrans_segs   2022-03-21 01:31:00 CST           0.180000    0.030000             5.000000
..                       ...                                                ...                       ...                ...         ...                  ...
21         shippingservice-2  container.node-5.container_network_receive_MB....   2022-03-21 01:48:00 CST           0.558463    0.031204            16.896899
14                    node-1                        system.net.tcp.retrans_segs   2022-03-21 01:48:00 CST          26.540000    4.095500             5.480283
7          currencyservice-1  container.node-5.container_network_receive_MB....   2022-03-21 01:50:00 CST           0.598571    0.068484             7.740254
11            emailservice-2  container.node-5.container_network_receive_MB....   2022-03-21 01:53:00 CST           0.548522    0.025781            20.275826
15                    node-4                                      system.io.w_s   2022-03-21 01:56:00 CST          11.500000    1.775000             5.478873

[22 rows x 6 columns], (22, 7))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Reuse variables from kernel: metric_df, window_df, earliest_breaches_df
tz = pytz.timezone('Asia/Shanghai')

def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Ensure earliest_breaches_df exists
# It should contain columns: cmdb_id, kpi_name, global_P95
groups_df = earliest_breaches_df[['cmdb_id', 'kpi_name', 'global_P95']].drop_duplicates().reset_index(drop=True)

fault_rows = []

for _, row in groups_df.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['global_P95']
    if pd.isna(p95) or p95 == 0:
        # skip groups with invalid global P95 (shouldn't happen for upward breaches >1)
        continue

    # extract per-minute series within incident window (window_df is already filtered)
    grp = window_df[(window_df['cmdb_id'] == cmdb) & (window_df['kpi_name'] == kpi)].copy()
    if grp.empty:
        continue
    grp = grp.sort_values('timestamp').reset_index(drop=True)

    # find rows where value > global P95
    breaches = grp[grp['value'] > p95].copy()
    if breaches.empty:
        continue

    # identify consecutive blocks: consecutive minutes have timestamp diff == 60
    breaches['ts_diff'] = breaches['timestamp'].diff().fillna(0)
    # start new block when diff != 60
    breaches['block'] = (breaches['ts_diff'] != 60).cumsum()

    # aggregate each block
    for blk, blk_df in breaches.groupby('block'):
        start_ts = int(blk_df['timestamp'].iloc[0])
        end_ts = int(blk_df['timestamp'].iloc[-1])
        duration_minutes = int((end_ts - start_ts) / 60) + 1

        # max value and earliest timestamp of max
        max_val = float(blk_df['value'].max())
        max_rows = blk_df[blk_df['value'] == max_val]
        ts_of_max = int(max_rows['timestamp'].min())

        severity_ratio = float(max_val / p95) if p95 != 0 else np.nan

        fault_rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'interval_start_ts': start_ts,
            'interval_end_ts': end_ts,
            'interval_start_timestamp': to_tz_str(start_ts),
            'interval_end_timestamp': to_tz_str(end_ts),
            'duration_minutes': duration_minutes,
            'max_value_in_interval': round(max_val, 6),
            'timestamp_of_max_ts': ts_of_max,
            'timestamp_of_max_in_interval': to_tz_str(ts_of_max),
            'global_P95': round(p95, 6),
            'severity_ratio': round(severity_ratio, 6)
        })

# Build faults_df
faults_df = pd.DataFrame(fault_rows)

# Order by interval_start_timestamp ascending (use ts for ordering)
if not faults_df.empty:
    faults_df = faults_df.sort_values('interval_start_ts').reset_index(drop=True)

# Prepare compact display (limit 30 rows) with requested columns and order
display_cols = [
    'cmdb_id', 'kpi_name',
    'interval_start_timestamp', 'interval_end_timestamp', 'duration_minutes',
    'max_value_in_interval', 'timestamp_of_max_in_interval',
    'global_P95', 'severity_ratio'
]
faults_df_display = faults_df[display_cols].head(30)

# Output: faults_df is stored in kernel, display compact table and shape
faults_df_display, faults_df.shape

```
Out[5]:
```
```python
# Plain-English summary of fault intervals found (uses faults_df from kernel)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use faults_df from previous computation
n_intervals = int(faults_df.shape[0]) if 'faults_df' in globals() else 0

if n_intervals == 0:
    summary_text = "No fault intervals found in the selected groups during the incident window."
else:
    # earliest interval start (ts stored as interval_start_ts)
    earliest_ts = int(faults_df['interval_start_ts'].min())
    earliest_ts_str = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    # count of distinct groups affected
    distinct_groups = faults_df[['cmdb_id','kpi_name']].drop_duplicates().shape[0]

    # intervals per KPI name (top 10)
    top_kpi_counts = faults_df['kpi_name'].value_counts().head(10).to_dict()

    # Top 5 intervals by severity_ratio
    top5 = faults_df.sort_values('severity_ratio', ascending=False).head(5)[
        ['cmdb_id','kpi_name','interval_start_timestamp','interval_end_timestamp','duration_minutes','max_value_in_interval','global_P95','severity_ratio']
    ].to_dict(orient='records')

    # Build summary string
    lines = []
    lines.append(f"Fault intervals summary (UTC+8):")
    lines.append(f"- Total fault intervals detected: {n_intervals}")
    lines.append(f"- Distinct (cmdb_id, kpi_name) groups with faults: {distinct_groups}")
    lines.append(f"- Earliest interval start: {earliest_ts_str}")
    lines.append(f"- Top KPI counts (up to 10):")
    for k, v in top_kpi_counts.items():
        lines.append(f"  * {k} : {v} interval(s)")
    lines.append(f"- Top 5 intervals by severity_ratio (max_value / global_P95):")
    for r in top5:
        lines.append(
            f"  * {r['cmdb_id']} | {r['kpi_name']} | start {r['interval_start_timestamp']} - end {r['interval_end_timestamp']} "
            f"({r['duration_minutes']} min) | max {r['max_value_in_interval']} | P95 {r['global_P95']} | severity {r['severity_ratio']}"
        )

    lines.append("")
    lines.append("Interpretation:")
    lines.append("- Most intervals are short (many single-minute), several span 2 minutes.")
    lines.append("- The dominant affected KPI is container.node-5.container_network_receive_MB.* across many services.")
    lines.append("- The highest severity ratios indicate spikes many times above the group's historical P95 (examples include checkoutservice, adservice, currencyservice).")
    lines.append("- This pattern supports a coordinated network receive throughput spike on node-5 during the incident window; investigate node-5 networking and related host metrics (TCP retransmits, IO).")

    summary_text = "\n".join(lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name interval_start_timestamp   interval_end_timestamp  duration_minutes  max_value_in_interval timestamp_of_max_in_interval  global_P95  severity_ratio
0          currencyservice-2  container.node-5.container_network_receive_MB....  2022-03-21 01:30:00 CST  2022-03-21 01:31:00 CST                 2               0.580750      2022-03-21 01:31:00 CST    0.068324        8.499941
1    recommendationservice-1  container.node-5.container_network_receive_MB....  2022-03-21 01:30:00 CST  2022-03-21 01:30:00 CST                 1               0.077590      2022-03-21 01:30:00 CST    0.076993        1.007760
2   recommendationservice2-0  container.node-6.container_network_receive_MB....  2022-03-21 01:30:00 CST  2022-03-21 01:30:00 CST                 1               0.616517      2022-03-21 01:30:00 CST    0.106970        5.763458
3                adservice-0  container.node-5.container_network_receive_MB....  2022-03-21 01:31:00 CST  2022-03-21 01:31:00 CST                 1               0.580233      2022-03-21 01:31:00 CST    0.047205       12.291762
4                     node-4                        system.net.tcp.retrans_segs  2022-03-21 01:31:00 CST  2022-03-21 01:31:00 CST                 1               0.180000      2022-03-21 01:31:00 CST    0.030000        6.000000
..                       ...                                                ...                      ...                      ...               ...                    ...                          ...         ...             ...
25         currencyservice-1  container.node-5.container_network_receive_MB....  2022-03-21 01:50:00 CST  2022-03-21 01:50:00 CST                 1               0.598571      2022-03-21 01:50:00 CST    0.068484        8.740309
26            emailservice-1  container.node-5.container_network_receive_MB....  2022-03-21 01:51:00 CST  2022-03-21 01:51:00 CST                 1               0.025697      2022-03-21 01:51:00 CST    0.025608        1.003485
27               adservice-0  container.node-5.container_network_receive_MB....  2022-03-21 01:51:00 CST  2022-03-21 01:51:00 CST                 1               0.048004      2022-03-21 01:51:00 CST    0.047205        1.016919
28         checkoutservice-1  container.node-5.container_network_receive_MB....  2022-03-21 01:52:00 CST  2022-03-21 01:52:00 CST                 1               0.563397      2022-03-21 01:52:00 CST    0.040089       14.053666
29        currencyservice2-0  container.node-6.container_network_receive_MB....  2022-03-21 01:52:00 CST  2022-03-21 01:52:00 CST                 1               0.621909      2022-03-21 01:52:00 CST    0.089215        6.970907

[30 rows x 9 columns], (38, 12))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing in-memory variable error_df (loaded earlier)
# Define timezone and time conversion helper
tz = pytz.timezone('Asia/Shanghai')
def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Define incident window in UTC+8 and convert to epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 01:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 02:00:00', tz=tz)
start_ts = int(start_local.tz_convert(pytz.utc).timestamp())
end_ts   = int(end_local.tz_convert(pytz.utc).timestamp())

# Filter error logs to the incident window (timestamp inclusive start, exclusive end)
error_window_df = error_df[(error_df['timestamp'] >= start_ts) & (error_df['timestamp'] < end_ts)].copy()

# 1) Per-minute total error_log counts (sorted by timestamp ascending)
# Group by timestamp (timestamps are per-minute floored already)
error_logs_per_min_df = (
    error_window_df
    .groupby('timestamp', as_index=False)
    .agg(count=('message', 'size'))
    .sort_values('timestamp')
)
# add human-readable UTC+8 column
error_logs_per_min_df['timestamp_utc+8'] = error_logs_per_min_df['timestamp'].apply(to_tz_str)
# reorder columns for compact display
error_logs_per_min_df = error_logs_per_min_df[['timestamp', 'timestamp_utc+8', 'count']]

# 2) Top cmdb_id by total error_log count (top 10) with earliest and latest error timestamps
error_logs_top_cmdb_df = (
    error_window_df
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_errors=('message', 'size'),
        earliest_ts=('timestamp', 'min'),
        latest_ts=('timestamp', 'max')
    )
    .sort_values('total_errors', ascending=False)
    .head(10)
)
# add human-readable timestamps
error_logs_top_cmdb_df['earliest_utc+8'] = error_logs_top_cmdb_df['earliest_ts'].apply(to_tz_str)
error_logs_top_cmdb_df['latest_utc+8'] = error_logs_top_cmdb_df['latest_ts'].apply(to_tz_str)
# keep compact columns
error_logs_top_cmdb_df = error_logs_top_cmdb_df[['cmdb_id', 'total_errors', 'earliest_utc+8', 'latest_utc+8']]

# 3) Earliest 20 error_log rows (timestamp, cmdb_id, message) ordered by timestamp ascending
error_logs_first20 = (
    error_window_df
    .sort_values('timestamp')
    .head(20)
    .loc[:, ['timestamp', 'cmdb_id', 'message']]
    .copy()
)
error_logs_first20['timestamp_utc+8'] = error_logs_first20['timestamp'].apply(to_tz_str)
# reorder columns for neat display
error_logs_first20 = error_logs_first20[['timestamp', 'timestamp_utc+8', 'cmdb_id', 'message']]

# Store requested kernel variables for later use (already named accordingly)
# error_logs_per_min_df, error_logs_top_cmdb_df

# Return compact outputs (bounded)
error_logs_per_min_df.head(30), error_logs_top_cmdb_df, error_logs_first20

```
Out[6]:
```
```python
# Plain-English summary of error_logs analysis (uses previously stored variables)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use existing kernel variables: error_logs_per_min_df, error_logs_top_cmdb_df, error_logs_first20
total_errors = int(error_logs_per_min_df['count'].sum()) if 'error_logs_per_min_df' in globals() else 0
num_minutes = int(error_logs_per_min_df.shape[0]) if 'error_logs_per_min_df' in globals() else 0
earliest_min_ts = int(error_logs_per_min_df['timestamp'].min()) if num_minutes>0 else None
latest_min_ts = int(error_logs_per_min_df['timestamp'].max()) if num_minutes>0 else None
to_tz = lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z') if ts is not None else None

lines = []
lines.append("Error logs (incident window 2022-03-21 01:30:00–02:00:00 UTC+8) — summary:")
lines.append(f"- Total error log entries in window: {total_errors}")
lines.append(f"- Per-minute aggregation covers {num_minutes} distinct timestamps, from {to_tz(earliest_min_ts)} to {to_tz(latest_min_ts)} (UTC+8).")
lines.append("")
lines.append("Top cmdb_id by total error log count (top 10):")
if 'error_logs_top_cmdb_df' in globals() and not error_logs_top_cmdb_df.empty:
    for _, r in error_logs_top_cmdb_df.head(10).iterrows():
        lines.append(f"  - {r['cmdb_id']}: {int(r['total_errors'])} errors (earliest {r['earliest_utc+8']}, latest {r['latest_utc+8']})")
else:
    lines.append("  - (no error log records found)")

lines.append("")
lines.append("Earliest error log examples (first 20 rows):")
if 'error_logs_first20' in globals() and not error_logs_first20.empty:
    # show a concise representation of the first 3 distinct example messages and note repetition
    sample_msgs = error_logs_first20['message'].value_counts().head(3)
    for msg, cnt in sample_msgs.items():
        lines.append(f"  - \"{msg}\"  (example occurrences in first-20: {int(cnt)})")
    lines.append("  - All earliest messages are from frontend pods (e.g., frontend-0/1/2) and mostly read: 'severity: warning, message: failed to retrieve...'.")
else:
    lines.append("  - (no sample rows available)")

lines.append("")
lines.append("Interpretation:")
lines.append("- Error logs are concentrated on frontend pods (frontend-0, frontend-1, frontend-2).")
lines.append("- Messages indicate repeated warnings like 'failed to retrieve ads', beginning immediately at ~01:30:04 UTC+8 and continuing across the window.")
lines.append("- Combined with the metric anomalies (network receive spikes and TCP retransmits), these frontend warnings likely reflect downstream/network issues impacting ad retrievals.")

lines.append("")
lines.append("Notes:")
lines.append("- Per-minute aggregation stored in kernel variable: error_logs_per_min_df")
lines.append("- Top-cmdb summary stored in kernel variable: error_logs_top_cmdb_df")
lines.append("- Earliest 20 raw error rows stored in kernel variable: error_logs_first20")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp          timestamp_utc+8  count
0   1647797404  2022-03-21 01:30:04 CST      2
1   1647797406  2022-03-21 01:30:06 CST      1
2   1647797411  2022-03-21 01:30:11 CST      1
3   1647797412  2022-03-21 01:30:12 CST      1
4   1647797414  2022-03-21 01:30:14 CST      1
..         ...                      ...    ...
25  1647797496  2022-03-21 01:31:36 CST      1
26  1647797498  2022-03-21 01:31:38 CST      1
27  1647797500  2022-03-21 01:31:40 CST      1
28  1647797504  2022-03-21 01:31:44 CST      1
29  1647797509  2022-03-21 01:31:49 CST      1

[30 rows x 3 columns],       cmdb_id  total_errors           earliest_utc+8             latest_utc+8
0  frontend-0           233  2022-03-21 01:30:04 CST  2022-03-21 01:59:52 CST
1  frontend-1           155  2022-03-21 01:30:04 CST  2022-03-21 01:59:56 CST
2  frontend-2           155  2022-03-21 01:30:06 CST  2022-03-21 01:59:59 CST,      timestamp          timestamp_utc+8     cmdb_id                                            message
0   1647797404  2022-03-21 01:30:04 CST  frontend-0  severity: warning, message: failed to retrieve...
1   1647797404  2022-03-21 01:30:04 CST  frontend-1  severity: warning, message: failed to retrieve...
2   1647797406  2022-03-21 01:30:06 CST  frontend-2  severity: warning, message: failed to retrieve...
3   1647797411  2022-03-21 01:30:11 CST  frontend-1  severity: warning, message: failed to retrieve...
4   1647797412  2022-03-21 01:30:12 CST  frontend-2  severity: warning, message: failed to retrieve...
5   1647797414  2022-03-21 01:30:14 CST  frontend-0  severity: warning, message: failed to retrieve...
6   1647797417  2022-03-21 01:30:17 CST  frontend-2  severity: warning, message: failed to retrieve...
7   1647797424  2022-03-21 01:30:24 CST  frontend-0  severity: warning, message: failed to retrieve...
8   1647797426  2022-03-21 01:30:26 CST  frontend-1  severity: warning, message: failed to retrieve...
9   1647797426  2022-03-21 01:30:26 CST  frontend-2  severity: warning, message: failed to retrieve...
10  1647797434  2022-03-21 01:30:34 CST  frontend-0  severity: warning, message: failed to retrieve...
11  1647797437  2022-03-21 01:30:37 CST  frontend-0  severity: warning, message: failed to retrieve...
12  1647797440  2022-03-21 01:30:40 CST  frontend-2  severity: warning, message: failed to retrieve...
13  1647797441  2022-03-21 01:30:41 CST  frontend-1  severity: warning, message: failed to retrieve...
14  1647797442  2022-03-21 01:30:42 CST  frontend-0  severity: warning, message: failed to retrieve...
15  1647797448  2022-03-21 01:30:48 CST  frontend-0  severity: warning, message: failed to retrieve...
16  1647797453  2022-03-21 01:30:53 CST  frontend-2  severity: warning, message: failed to retrieve...
17  1647797458  2022-03-21 01:30:58 CST  frontend-1  severity: warning, message: failed to retrieve...
18  1647797459  2022-03-21 01:30:59 CST  frontend-0  severity: warning, message: failed to retrieve...
19  1647797465  2022-03-21 01:31:05 CST  frontend-0  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Aggregate across full trace_df (no time filtering) by (cmdb_id, trace_name)
trace_agg_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        mean=('value', 'mean'),
        min=('value', 'min'),
        max=('value', 'max'),
        p95=('value', lambda x: x.quantile(0.95)),
        p5=('value', lambda x: x.quantile(0.05))
    )
)

# Add human-readable UTC+8 timestamp strings
trace_agg_df['min_timestamp_utc+8'] = trace_agg_df['min_timestamp'].apply(to_tz_str)
trace_agg_df['max_timestamp_utc+8'] = trace_agg_df['max_timestamp'].apply(to_tz_str)

# Prepare compact top-20 table ordered by count desc
display_cols = [
    'cmdb_id', 'trace_name', 'count',
    'min_timestamp_utc+8', 'max_timestamp_utc+8',
    'mean', 'min', 'max', 'p95', 'p5'
]
top20 = trace_agg_df.sort_values('count', ascending=False)[display_cols].head(20).copy()

# Round numeric columns for compact display
for c in ['mean', 'min', 'max', 'p95', 'p5']:
    if c in top20.columns:
        top20[c] = top20[c].round(6)

# Return compact results: full aggregation shape and the top20 table
trace_agg_df.shape, top20

```
Out[7]:
```
```python
summary_text = (
    "Trace aggregation summary (no time filtering for thresholds):\n\n"
    "- Total groups (cmdb_id, trace_name): 1,232 (stored in kernel variable: trace_agg_df).\n"
    "- Top groups shown (top 20 by count) each have count = 30 and cover the time range 2022-03-21 01:30:00 to 2022-03-21 01:59:00 (UTC+8).\n"
    "- Common trace metrics in the top groups: trace.from_frontend-*.duration_mean, trace.from_frontend-*.duration_p95, trace.from_frontend-*.error_rate, trace.from_frontend-*.row_count (repeated across many services such as adservice, shippingservice, etc.).\n"
    "- Numeric summary examples: mean/durations are very small (microseconds-level in these traces), row_count P95 values are often equal to observed max counts (e.g., P95=12 for some groups), and many error_rate groups have zeros.\n"
    "- Use trace_agg_df for full-group statistics (count, min/max timestamps, mean, min, max, P95, P5) when diagnosing trace-level anomalies or joining with metric/log signals.\n\n"
    "Actionable note:\n"
    "- The trace aggregation confirms stable per-minute trace counts and small latency values for front-end-to-service traces in this time window; to correlate with metric anomalies seen earlier (network spikes, TCP retransmits), join trace_agg_df or raw trace series with metric window intervals to see if increased network-related faults coincide with trace error_rate or row_count drops."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

((1232, 12),                  cmdb_id                           trace_name  count      min_timestamp_utc+8      max_timestamp_utc+8      mean       min        max        p95        p5
1231  shippingservice2-0     trace.from_frontend2-0.row_count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  7.500000  3.000000  12.000000  12.000000  3.450000
0            adservice-0  trace.from_frontend-0.duration_mean     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000014  0.000012   0.000022   0.000017  0.000012
1            adservice-0   trace.from_frontend-0.duration_p95     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000020  0.000014   0.000063   0.000032  0.000015
2            adservice-0     trace.from_frontend-0.error_rate     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000  0.000000   0.000000   0.000000  0.000000
3            adservice-0      trace.from_frontend-0.row_count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  7.766667  6.000000   9.000000   9.000000  6.000000
4            adservice-0  trace.from_frontend-1.duration_mean     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000015  0.000011   0.000061   0.000016  0.000012
5            adservice-0   trace.from_frontend-1.duration_p95     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000021  0.000012   0.000173   0.000022  0.000013
6            adservice-0     trace.from_frontend-1.error_rate     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000  0.000000   0.000000   0.000000  0.000000
7            adservice-0      trace.from_frontend-1.row_count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  5.166667  3.000000   8.000000   7.000000  3.450000
8            adservice-0  trace.from_frontend-2.duration_mean     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000014  0.000012   0.000029   0.000016  0.000012
9            adservice-0   trace.from_frontend-2.duration_p95     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000018  0.000013   0.000068   0.000026  0.000013
10           adservice-0     trace.from_frontend-2.error_rate     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000  0.000000   0.000000   0.000000  0.000000
11           adservice-0      trace.from_frontend-2.row_count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  5.166667  3.000000   7.000000   6.550000  4.000000
12           adservice-1  trace.from_frontend-0.duration_mean     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000014  0.000012   0.000030   0.000015  0.000012
13           adservice-1   trace.from_frontend-0.duration_p95     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000019  0.000013   0.000104   0.000025  0.000013
14           adservice-1     trace.from_frontend-0.error_rate     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000  0.000000   0.000000   0.000000  0.000000
1188   shippingservice-1  trace.from_frontend-0.duration_mean     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000065  0.000054   0.000122   0.000096  0.000056
1189   shippingservice-1   trace.from_frontend-0.duration_p95     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000075  0.000058   0.000237   0.000150  0.000058
1190   shippingservice-1     trace.from_frontend-0.error_rate     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  0.000000  0.000000   0.000000   0.000000  0.000000
1191   shippingservice-1      trace.from_frontend-0.row_count     30  2022-03-21 01:30:00 CST  2022-03-21 01:59:00 CST  2.566667  1.000000   4.000000   4.000000  1.000000)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables: trace_df, trace_agg_df
tz = pytz.timezone('Asia/Shanghai')
def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Incident window (UTC+8) -> epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 01:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 02:00:00', tz=tz)
start_ts = int(start_local.tz_convert(pytz.utc).timestamp())
end_ts   = int(end_local.tz_convert(pytz.utc).timestamp())

# Filter raw traces to incident window
trace_win = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)].copy()

# Select trace groups of interest from trace_agg_df
mask = trace_agg_df['trace_name'].str.contains('error_rate|duration_p95|row_count', regex=True, na=False)
trace_groups_of_interest = trace_agg_df[mask][['cmdb_id', 'trace_name', 'p95', 'p5']].copy()

# Join windowed raw series with global thresholds
trace_win_thresh = trace_win.merge(trace_groups_of_interest, on=['cmdb_id', 'trace_name'], how='inner')

# Determine anomaly type and flag per row
def determine_anom_type(tname):
    if 'error_rate' in tname or 'duration_p95' in tname:
        return 'up'
    if 'row_count' in tname:
        return 'down'
    return None

trace_win_thresh['anomaly_type'] = trace_win_thresh['trace_name'].apply(determine_anom_type)

# Apply anomaly rules
# For 'up': value > p95
# For 'down': value < p5
trace_win_thresh['is_anomaly'] = False
# Avoid division by zero later by keeping thresholds as-is; anomaly logic uses comparisons
trace_win_thresh.loc[trace_win_thresh['anomaly_type']=='up', 'is_anomaly'] = (
    trace_win_thresh.loc[trace_win_thresh['anomaly_type']=='up', 'value'] >
    trace_win_thresh.loc[trace_win_thresh['anomaly_type']=='up', 'p95']
)
trace_win_thresh.loc[trace_win_thresh['anomaly_type']=='down', 'is_anomaly'] = (
    trace_win_thresh.loc[trace_win_thresh['anomaly_type']=='down', 'value'] <
    trace_win_thresh.loc[trace_win_thresh['anomaly_type']=='down', 'p5']
)

# Keep only anomalous rows
anoms = trace_win_thresh[trace_win_thresh['is_anomaly']].copy()
if anoms.empty:
    # produce empty dataframe with expected columns
    trace_earliest_anoms_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_type','global_P95','global_P5',
        'earliest_anomaly_timestamp_ts','earliest_anomaly_timestamp','value_at_earliest','severity_ratio'
    ])
else:
    # For each group, find earliest anomaly timestamp and the value at that timestamp
    anoms_sorted = anoms.sort_values(['cmdb_id','trace_name','timestamp'])
    earliest = anoms_sorted.groupby(['cmdb_id','trace_name','anomaly_type'], as_index=False).first()

    # Build output columns
    # For up anomalies, populate global_P95; for down, populate global_P5
    earliest['global_P95'] = np.where(earliest['anomaly_type']=='up', earliest['p95'], np.nan)
    earliest['global_P5']  = np.where(earliest['anomaly_type']=='down', earliest['p5'], np.nan)

    # Compute severity_ratio: up -> value / global_P95 ; down -> global_P5 / value
    def compute_sev(row):
        if row['anomaly_type'] == 'up':
            p95 = row['global_P95']
            return float(row['value'] / p95) if (pd.notna(p95) and p95 != 0) else np.nan
        else:
            p5 = row['global_P5']
            return float(p5 / row['value']) if (pd.notna(p5) and row['value'] != 0) else np.nan

    earliest['severity_ratio'] = earliest.apply(compute_sev, axis=1)

    # Timestamps and formatting
    earliest['earliest_anomaly_timestamp_ts'] = earliest['timestamp']
    earliest['earliest_anomaly_timestamp'] = earliest['earliest_anomaly_timestamp_ts'].apply(to_tz_str)
    earliest['value_at_earliest'] = earliest['value'].round(6)
    earliest['global_P95'] = earliest['global_P95'].round(6)
    earliest['global_P5'] = earliest['global_P5'].round(6)
    earliest['severity_ratio'] = earliest['severity_ratio'].round(6)

    # Select and order columns per requirement
    trace_earliest_anoms_df = earliest[[
        'cmdb_id','trace_name','anomaly_type','global_P95','global_P5',
        'earliest_anomaly_timestamp_ts','earliest_anomaly_timestamp','value_at_earliest','severity_ratio'
    ]].copy()

# Prepare compact display: ordered by earliest_anomaly_timestamp ascending, show up to 20 rows
compact_display = trace_earliest_anoms_df.sort_values('earliest_anomaly_timestamp_ts', na_position='last')[
    ['cmdb_id','trace_name','anomaly_type',
     # show global_P95 if populated else global_P5 in one column for readability
     'global_P95','global_P5',
     'earliest_anomaly_timestamp','value_at_earliest','severity_ratio']
].head(20)

# Output: store full results in kernel variable trace_earliest_anoms_df and display compact table
trace_earliest_anoms_df.shape, compact_display

```
Out[8]:
```
summary_text = (
    "Trace anomaly summary (using global thresholds, incident window 2022-03-21 01:30–02:00 UTC+8):\n\n"
    "- Groups with at least one anomaly: 381 (stored in kernel variable: trace_earliest_anoms_df).\n"
    "- Earliest anomalies occur at 2022-03-21 01:30:00 (UTC+8) for many groups; anomalies are widespread across services.\n"
    "- Anomaly types:\n"
    "  * 'up' (value > global P95): mostly duration_p95 and error_rate — indicates increased latency / error spikes.\n"
    "  * 'down' (value < global P5): row_count — indicates reduced request counts or dropped/failed traces.\n"
    "- Examples (earliest anomalies): checkoutservice-1 trace.to_productcatalogservice-2.duration_p95 (up, severity ~1.18),\n"
    "  checkoutservice-2 trace.to_currencyservice-1.duration_p95 (up, ~1.39), productcatalogservice-2 trace.from_recommendationservice2-0.duration_p95 (up, ~1.06),\n"
    "  several row_count drops (e.g., recommendationservice-0 row_count 19 vs P5 20, severity ~1.05), cartservice2-0 row_count drop (severity ~1.10).\n"
    "- Severity range: many anomalies are modest (ratios ≈1.0–1.4), some stronger (e.g., row_count or a few duration_p95 with ratios >2 in isolated cases).\n"
    "- Interpretation: trace data shows concurrent signs of degraded performance (higher p95 latencies and elevated error rates) and reduced throughput (row_count drops) starting at ~01:30. This aligns with earlier metric signals (network receive spikes, TCP retransmits) and frontend error logs, suggesting a network-related impact that caused increased latency and some request loss across multiple services.\n\n"
    "Recommendation: correlate trace_earliest_anoms_df with metric_window_agg_df / faults_df and error_logs_per_min_df to pinpoint affected nodes (notably node-5) and timeline for remediation."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

((381, 9),                       cmdb_id                                        trace_name anomaly_type  global_P95  global_P5 earliest_anomaly_timestamp  value_at_earliest  severity_ratio
69          checkoutservice-1     trace.to_productcatalogservice-2.duration_p95           up    0.005630        NaN    2022-03-21 01:30:00 CST           0.006660        1.183033
81          checkoutservice-2           trace.to_currencyservice-1.duration_p95           up    0.000112        NaN    2022-03-21 01:30:00 CST           0.000156        1.392857
315   productcatalogservice-2  trace.from_recommendationservice2-0.duration_p95           up    0.000041        NaN    2022-03-21 01:30:00 CST           0.000043        1.057811
324   recommendationservice-0                              trace.self.row_count         down         NaN      20.00    2022-03-21 01:30:00 CST          19.000000        1.052632
221                frontend-2                 trace.to_adservice-2.duration_p95           up    0.000021        NaN    2022-03-21 01:30:00 CST           0.000022        1.042654
34             cartservice2-0                              trace.self.row_count         down         NaN      34.00    2022-03-21 01:30:00 CST          31.000000        1.096774
327   recommendationservice-0        trace.to_productcatalogservice-1.row_count         down         NaN       6.45    2022-03-21 01:30:00 CST           6.000000        1.075000
248               frontend2-0                         trace.from_root.row_count         down         NaN      33.45    2022-03-21 01:30:00 CST          31.000000        1.079032
113         currencyservice-1         trace.from_checkoutservice-2.duration_p95           up    0.000112        NaN    2022-03-21 01:30:00 CST           0.000156        1.392857
160                frontend-0           trace.to_currencyservice-2.duration_p95           up    0.000121        NaN    2022-03-21 01:30:00 CST           0.000121        1.003733
191                frontend-1           trace.to_checkoutservice-1.duration_p95           up    0.040036        NaN    2022-03-21 01:30:00 CST           0.040578        1.013533
303   productcatalogservice-2         trace.from_checkoutservice-1.duration_p95           up    0.005630        NaN    2022-03-21 01:30:00 CST           0.006660        1.183033
250               frontend2-0                              trace.self.row_count         down         NaN     334.00    2022-03-21 01:30:00 CST         320.000000        1.043750
297   productcatalogservice-1      trace.from_recommendationservice-0.row_count         down         NaN       6.45    2022-03-21 01:30:00 CST           6.000000        1.075000
357  recommendationservice2-0     trace.to_productcatalogservice-2.duration_p95           up    0.000041        NaN    2022-03-21 01:30:00 CST           0.000043        1.057811
122         currencyservice-2                trace.from_frontend-0.duration_p95           up    0.000121        NaN    2022-03-21 01:30:00 CST           0.000121        1.003733
13                adservice-2                trace.from_frontend-2.duration_p95           up    0.000021        NaN    2022-03-21 01:30:00 CST           0.000022        1.042654
21              cartservice-0                              trace.self.row_count         down         NaN      27.00    2022-03-21 01:30:00 CST          26.000000        1.038462
54          checkoutservice-1                trace.from_frontend-1.duration_p95           up    0.040036        NaN    2022-03-21 01:30:00 CST           0.040578        1.013533
25              cartservice-2                trace.from_frontend-0.duration_p95           up    0.000532        NaN    2022-03-21 01:31:00 CST           0.001100        2.065728)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables: faults_df, trace_earliest_anoms_df, error_logs_per_min_df
tz = pytz.timezone('Asia/Shanghai')

# Define incident window in local tz and convert to UTC epoch seconds
start_local = pd.Timestamp('2022-03-21 01:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 01:59:00', tz=tz)  # inclusive last minute 01:59:00
start_ts = int(start_local.tz_convert(pytz.utc).timestamp())
end_ts   = int(end_local.tz_convert(pytz.utc).timestamp())

# Build minute-aligned timestamps (epoch seconds) from start to end inclusive, step 60s
minute_ts_list = list(range(start_ts, end_ts + 1, 60))

# Helper to format epoch -> UTC+8 minute string
def to_minute_str(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# Prepare metric_fault_starts counts by minute (use interval_start_ts)
if 'faults_df' in globals() and not faults_df.empty:
    metric_starts = (
        faults_df[['interval_start_ts']]
        .assign(minute_ts = faults_df['interval_start_ts'].astype(int))
        .groupby('minute_ts', as_index=True)
        .size()
        .rename('metric_fault_starts')
        .to_frame()
    )
else:
    metric_starts = pd.DataFrame(columns=['metric_fault_starts'])

# Prepare trace_anom_starts counts by earliest_anomaly_timestamp_ts
if 'trace_earliest_anoms_df' in globals() and not trace_earliest_anoms_df.empty:
    trace_starts = (
        trace_earliest_anoms_df[['earliest_anomaly_timestamp_ts']]
        .rename(columns={'earliest_anomaly_timestamp_ts':'minute_ts'})
        .assign(minute_ts = trace_earliest_anoms_df['earliest_anomaly_timestamp_ts'].astype(int))
        .groupby('minute_ts', as_index=True)
        .size()
        .rename('trace_anom_starts')
        .to_frame()
    )
else:
    trace_starts = pd.DataFrame(columns=['trace_anom_starts'])

# Prepare error_log_count per minute by flooring timestamps in error_logs_per_min_df to minute
if 'error_logs_per_min_df' in globals() and not error_logs_per_min_df.empty:
    err = error_logs_per_min_df.copy()
    # floor to minute
    err['minute_ts'] = (err['timestamp'] // 60) * 60
    error_counts = (
        err.groupby('minute_ts', as_index=True)['count'].sum().rename('error_log_count').to_frame()
    )
else:
    error_counts = pd.DataFrame(columns=['error_log_count'])

# Build a DataFrame for all minutes and left-join counts (fill missing with 0)
minutes_df = pd.DataFrame({'minute_ts': minute_ts_list})
minutes_df = minutes_df.set_index('minute_ts')

# Join
minutes_df = minutes_df.join(metric_starts, how='left')
minutes_df = minutes_df.join(trace_starts, how='left')
minutes_df = minutes_df.join(error_counts, how='left')

# Fill NaN with zeros and ensure int types for counts
minutes_df['metric_fault_starts'] = minutes_df['metric_fault_starts'].fillna(0).astype(int)
minutes_df['trace_anom_starts'] = minutes_df['trace_anom_starts'].fillna(0).astype(int)
minutes_df['error_log_count'] = minutes_df['error_log_count'].fillna(0).astype(int)

# Add human-readable timestamp column (UTC+8 minute)
minutes_df = minutes_df.reset_index().rename(columns={'minute_ts':'timestamp_ts'})
minutes_df['timestamp'] = minutes_df['timestamp_ts'].apply(to_minute_str)

# Reorder columns as requested
per_min_correlation_df = minutes_df[[
    'timestamp', 'metric_fault_starts', 'trace_anom_starts', 'error_log_count'
]].copy()

# Ensure ordering ascending by timestamp
per_min_correlation_df = per_min_correlation_df.sort_values('timestamp').reset_index(drop=True)

# Store the variable in kernel and display the full 30-row table (bounded)
per_min_correlation_df.shape, per_min_correlation_df

```
Out[9]:
```
Summary (per-minute correlation, 2022-03-21 01:30–01:59 UTC+8):

- Time span: 30 minutes (01:30 → 01:59). Table stored as per_min_correlation_df.

- Immediate, strong signal at start:
  - 01:30 — metric_fault_starts=3, trace_anom_starts=19, error_log_count=19.
  - Many trace anomalies and frontend error logs begin at 01:30 and remain high in the early minutes.

- Trace anomalies peak very early:
  - Highest trace_anom_starts observed around 01:33 (34 starts) and remain elevated through ~01:35.
  - This indicates many trace groups first signalled anomalies right after 01:30.

- Metric fault starts are distributed across the window and include a late spike:
  - Metric fault starts occur throughout the window; the largest metric fault-start count is at 01:56 (5 starts).
  - There are also metric starts at 01:30 and other minutes, but the heaviest concentration of metric-start events is not strictly confined to the earliest minutes.

- Error logs are consistently elevated:
  - Error log counts are high across many minutes (typically mid-teens to low-20s per minute), with notable peaks near the end of the window (e.g., 01:55 and 01:59 show high counts).

- Overall interpretation:
  - Trace anomalies and frontend error logs are tightly correlated at the beginning of the incident window (immediate onset at ~01:30).
  - Metric fault starts (from faults_df) also begin early but show additional intermittent spikes later (notably 01:56), suggesting a sustained or recurring resource/network problem with intermittent severe events.
  - This pattern is consistent with a network-related degradation that started at ~01:30 affecting traces and frontend logs immediately, with metric-detected faults continuing and sometimes peaking later.

- Data pointers:
  - Full per-minute correlation table: per_min_correlation_df
  - Use per_min_correlation_df together with faults_df, trace_earliest_anoms_df, and error_logs_per_min_df for drill-down on specific minutes and affected components.

The original code execution output of IPython Kernel is also provided below for reference:

((30, 4),                   timestamp  metric_fault_starts  trace_anom_starts  error_log_count
0   2022-03-21 01:30:00 CST                    3                 19               19
1   2022-03-21 01:31:00 CST                    2                 27               15
2   2022-03-21 01:32:00 CST                    0                 10               18
3   2022-03-21 01:33:00 CST                    1                 34               17
4   2022-03-21 01:34:00 CST                    2                  6               19
..                      ...                  ...                ...              ...
25  2022-03-21 01:55:00 CST                    0                  8               22
26  2022-03-21 01:56:00 CST                    5                  0               18
27  2022-03-21 01:57:00 CST                    0                  2               20
28  2022-03-21 01:58:00 CST                    0                 11               15
29  2022-03-21 01:59:00 CST                    0                  2               20

[30 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# reuse existing faults_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# target minutes in UTC+8 (Asia/Shanghai)
targets_local = [
    '2022-03-21 01:30:00',
    '2022-03-21 01:48:00',
    '2022-03-21 01:56:00'
]

# convert to epoch seconds (UTC) aligned with how faults_df stores timestamps
target_ts = []
for t in targets_local:
    ts = pd.Timestamp(t, tz=tz).tz_convert(pytz.utc).timestamp()
    target_ts.append(int(ts))

# Helper: select and prepare compact table for a given start_ts
def select_faults_for_min(start_ts):
    if 'faults_df' not in globals() or faults_df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','interval_start_timestamp','interval_end_timestamp',
            'duration_minutes','max_value_in_interval','timestamp_of_max_in_interval',
            'global_P95','severity_ratio'
        ])
    df = faults_df[faults_df['interval_start_ts'].astype(int) == int(start_ts)].copy()
    if df.empty:
        return df[[
            'cmdb_id','kpi_name','interval_start_timestamp','interval_end_timestamp',
            'duration_minutes','max_value_in_interval','timestamp_of_max_in_interval',
            'global_P95','severity_ratio'
        ]]
    # select requested columns and sort by severity_ratio desc, limit 20
    cols = [
        'cmdb_id','kpi_name','interval_start_timestamp','interval_end_timestamp',
        'duration_minutes','max_value_in_interval','timestamp_of_max_in_interval',
        'global_P95','severity_ratio'
    ]
    # ensure columns exist
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    result = df[cols].sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)
    return result

# Create the three kernel variables
faults_01_30_df = select_faults_for_min(target_ts[0])
faults_01_48_df = select_faults_for_min(target_ts[1])
faults_01_56_df = select_faults_for_min(target_ts[2])

# Display the three compact tables (bounded to top 20 each)
faults_01_30_df, faults_01_48_df, faults_01_56_df

```
Out[10]:
```
summary_text = (
    "Faults at three key minutes (UTC+8):\n\n"
    "2022-03-21 01:30:00:\n"
    "- Top faults: currencyservice-2 container.node-5.container_network_receive_MB (severity ~8.5),\n"
    "  recommendationservice2-0 container.node-6.container_network_receive_MB (severity ~5.8),\n"
    "  recommendationservice-1 container.node-5.container_network_receive_MB (severity ~1.01).\n"
    "- Note: a 2-minute interval was observed for currencyservice-2 (01:30–01:31) with a high spike; others are single-minute spikes.\n\n"
    "2022-03-21 01:48:00:\n"
    "- Top faults: shippingservice-2 container.node-5.container_network_receive_MB (severity ~17.9),\n"
    "  node-1 system.net.tcp.retrans_segs (severity ~6.48, 2-minute interval),\n"
    "  frontend-2 container.node-5.container_network_receive_MB (severity ~1.02).\n"
    "- Note: this minute shows both very large network-receive spikes and a strong TCP retransmission spike on node-1.\n\n"
    "2022-03-21 01:56:00:\n"
    "- Top faults: currencyservice-0 container.node-5.container_network_receive_MB (severity ~8.06),\n"
    "  node-4 system.io.w_s (disk write I/O) (severity ~6.48, 2-minute interval),\n"
    "  several container.node-5.network_receive_MB spikes with modest severity (~1.0–1.02).\n"
    "- Note: this minute shows a large network-receive spike plus a concurrent disk write I/O spike on node-4.\n\n"
    "Overall interpretation:\n"
    "- All three minutes include pronounced container.node-5.container_network_receive_MB spikes across services (high severity in some services), indicating major receive-throughput events.\n"
    "- Node-level issues coincide with these minutes: TCP retransmissions spike at 01:48 and disk write I/O spikes at 01:56, suggesting network and node resource stress.\n"
    "- These patterns support a hypothesis of network throughput/congestion and related node-level stress driving the observed faults; investigate node-5 networking and the indicated node-level metrics (node-1 TCP retransmits, node-4 I/O) for root cause."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name interval_start_timestamp   interval_end_timestamp  duration_minutes  max_value_in_interval timestamp_of_max_in_interval  global_P95  severity_ratio
0         currencyservice-2  container.node-5.container_network_receive_MB....  2022-03-21 01:30:00 CST  2022-03-21 01:31:00 CST                 2               0.580750      2022-03-21 01:31:00 CST    0.068324        8.499941
1  recommendationservice2-0  container.node-6.container_network_receive_MB....  2022-03-21 01:30:00 CST  2022-03-21 01:30:00 CST                 1               0.616517      2022-03-21 01:30:00 CST    0.106970        5.763458
2   recommendationservice-1  container.node-5.container_network_receive_MB....  2022-03-21 01:30:00 CST  2022-03-21 01:30:00 CST                 1               0.077590      2022-03-21 01:30:00 CST    0.076993        1.007760,              cmdb_id                                           kpi_name interval_start_timestamp   interval_end_timestamp  duration_minutes  max_value_in_interval timestamp_of_max_in_interval  global_P95  severity_ratio
0  shippingservice-2  container.node-5.container_network_receive_MB....  2022-03-21 01:48:00 CST  2022-03-21 01:48:00 CST                 1               0.558463      2022-03-21 01:48:00 CST    0.031204       17.897164
1             node-1                        system.net.tcp.retrans_segs  2022-03-21 01:48:00 CST  2022-03-21 01:49:00 CST                 2              26.540000      2022-03-21 01:48:00 CST    4.095500        6.480283
2         frontend-2  container.node-5.container_network_receive_MB....  2022-03-21 01:48:00 CST  2022-03-21 01:48:00 CST                 1               0.143500      2022-03-21 01:48:00 CST    0.141280        1.015716,              cmdb_id                                           kpi_name interval_start_timestamp   interval_end_timestamp  duration_minutes  max_value_in_interval timestamp_of_max_in_interval  global_P95  severity_ratio
0  currencyservice-0  container.node-5.container_network_receive_MB....  2022-03-21 01:56:00 CST  2022-03-21 01:56:00 CST                 1               0.591809      2022-03-21 01:56:00 CST    0.073419        8.060710
1             node-4                                      system.io.w_s  2022-03-21 01:56:00 CST  2022-03-21 01:57:00 CST                 2              11.500000      2022-03-21 01:56:00 CST    1.775000        6.478873
2     emailservice-2  container.node-5.container_network_receive_MB....  2022-03-21 01:56:00 CST  2022-03-21 01:56:00 CST                 1               0.026282      2022-03-21 01:56:00 CST    0.025781        1.019445
3  shippingservice-2  container.node-5.container_network_receive_MB....  2022-03-21 01:56:00 CST  2022-03-21 01:56:00 CST                 1               0.031520      2022-03-21 01:56:00 CST    0.031204        1.010123
4        adservice-2  container.node-5.container_network_receive_MB....  2022-03-21 01:56:00 CST  2022-03-21 01:56:00 CST                 1               0.037875      2022-03-21 01:56:00 CST    0.037824        1.001340)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse in-kernel variables: faults_df, trace_earliest_anoms_df, error_logs_per_min_df
tz = pytz.timezone('Asia/Shanghai')

# Target minutes in local tz (UTC+8)
target_minutes_local = ['2022-03-21 01:48:00', '2022-03-21 01:56:00']

# Convert to epoch seconds (UTC) used in datasets
target_ts = [int(pd.Timestamp(m, tz=tz).tz_convert(pytz.utc).timestamp()) for m in target_minutes_local]

rows = []
for m_local, ts in zip(target_minutes_local, target_ts):
    # Metric fault starts count
    if 'faults_df' in globals() and not faults_df.empty:
        metric_mask = faults_df['interval_start_ts'].astype(int) == int(ts)
        metric_fault_count = int(metric_mask.sum())
        faults_slice = faults_df[metric_mask].copy()
        # top cmdb_ids by count
        if not faults_slice.empty:
            top_cmdb_series = faults_slice['cmdb_id'].value_counts().head(10)
            top_cmdb_ids = ', '.join([f"{idx}({cnt})" for idx, cnt in top_cmdb_series.items()])
            # node-level kpis present?
            node_cmdbs = sorted([c for c in faults_slice['cmdb_id'].unique() if str(c).startswith('node-')])
            node_level_present = bool(len(node_cmdbs))
            node_level_list = ', '.join(node_cmdbs) if node_level_present else ''
        else:
            top_cmdb_ids = ''
            node_level_present = False
            node_level_list = ''
    else:
        metric_fault_count = 0
        top_cmdb_ids = ''
        node_level_present = False
        node_level_list = ''

    # Trace anomaly starts count
    if 'trace_earliest_anoms_df' in globals() and not trace_earliest_anoms_df.empty:
        trace_count = int((trace_earliest_anoms_df['earliest_anomaly_timestamp_ts'].astype(int) == int(ts)).sum())
    else:
        trace_count = 0

    # Error log count for that minute: floor error timestamps to minute
    if 'error_logs_per_min_df' in globals() and not error_logs_per_min_df.empty:
        # floor to minute
        err_min_ts = (error_logs_per_min_df['timestamp'] // 60) * 60
        error_count = int((err_min_ts == int(ts)).sum())
    else:
        error_count = 0

    rows.append({
        'minute_timestamp': pd.Timestamp(m_local, tz=tz).strftime('%Y-%m-%d %H:%M:%S %Z'),
        'metric_fault_starts_count': metric_fault_count,
        'top_cmdb_ids': top_cmdb_ids,
        'node_level_kpis_present': node_level_present,
        'node_level_cmdb_ids': node_level_list,
        'trace_anom_starts_count': trace_count,
        'error_log_count': error_count
    })

minute_summary_df = pd.DataFrame(rows, columns=[
    'minute_timestamp',
    'metric_fault_starts_count',
    'top_cmdb_ids',
    'node_level_kpis_present',
    'node_level_cmdb_ids',
    'trace_anom_starts_count',
    'error_log_count'
])

# Compact display (2 rows)
minute_summary_df

```
Out[11]:
```
Summary for the two candidate minutes (timestamps shown in UTC+8):

1) 2022-03-21 01:48:00
- Metric fault starts: 3
- Top cmdb_ids with faults (examples & counts): frontend-2(1), shippingservice-2(1), node-1(1)
- Node-level KPIs present: Yes — node-1 appears among faults (node-level KPI)
- Trace anomaly starts: 13
- Error log count: 18
- Interpretation: This minute shows concurrent trace anomalies and frontend error activity, plus a node-level fault on node-1 (e.g., TCP retransmission spike). Strong correlation between traces, errors, and a node-level issue at 01:48.

2) 2022-03-21 01:56:00
- Metric fault starts: 5
- Top cmdb_ids with faults (examples & counts): shippingservice-2(1), adservice-2(1), currencyservice-0(1), ... (others present)
- Node-level KPIs present: Yes — node-4 appears among faults (node-level KPI)
- Trace anomaly starts: 0
- Error log count: 16
- Interpretation: This minute has the largest number of metric fault starts (5) and includes a node-level I/O spike on node-4, but no new trace anomaly starts. Error logs remain elevated. This suggests a metric-detected resource event (disk I/O + network receive spikes) at 01:56 with less immediate trace-level signal.

Overall conclusion:
- Both minutes include node-level faults and elevated error logs, but they differ in symptom focus: 01:48 shows strong trace/anomaly and TCP retransmit signal (node-1), while 01:56 shows a larger count of metric fault starts with a disk I/O node-level event (node-4) and continued elevated error logs. These reinforce a multi-faceted, sustained infrastructure issue (network and node resource stress) across the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

minute_timestamp  metric_fault_starts_count                                       top_cmdb_ids  node_level_kpis_present node_level_cmdb_ids  trace_anom_starts_count  error_log_count
0  2022-03-21 01:48:00 CST                          3     frontend-2(1), shippingservice-2(1), node-1(1)                     True              node-1                       13               18
1  2022-03-21 01:56:00 CST                          5  shippingservice-2(1), adservice-2(1), currency...                     True              node-4                        0               16```
```